# 概念知识整理：逐算子交互调试

输入为 datasets 采集数据；不读取旧 clean_docs。每个执行 cell 使用真实业务算子，经 `demiflow.map_async` 执行，不调用 CLI 拼装截图。

前半段真实运行来源检查、扫描、原始材料包、清洗；后半段 identity → export 使用清洗结果，模型调用默认关闭。清洗是保守初版：排除明确导航，保留未知模板并标记；清洗与候选导出不等于知识库质量验收。

使用公共 Python 内核 `/yzp/zhaozy/yangzepeng/0905/env/bin/python`。按顺序执行；随后可反复执行任意查看 cell。显示 limit／sample 不改变实际输入。源码、配置或算子输入变化时换 RUN_NAME，代码修改后重启内核；旧请求和结果不覆盖。保存 notebook 后再启动运行，执行输出不参与版本哈希。

这三个概念是定向工程案例；扫描预算与查看条数分别设置。不是全库随机抽样。

## 阅读导航：每个算子的具体动作

```text
检查来源文件
  → 按名称读取概念信息，按QID读取语言页面ID
  → 按概念名／QID／页面ID查找文档和图片
  → 读取正文、核对图片字节、按请求汇集
  → 保存原始材料包
  → 清除明确外壳，记录原文块定位
  → 判定概念歧义，筛选可关联材料
  → 去重、按预算选取、截取正文、准备图片字节
  → 提取带引文知识
  → 对照原文修订，记录未解冲突
  → 逐图逐知识判断支持
  → 保存候选与阻塞记录
```

| 步骤 | 动作与对象 | 本批状态 |
| --- | --- | --- |
| 1 | 检查来源文件是否存在、样本能否解析 | 已执行 |
| 2 | 按名称读取概念信息，按QID读取语言页面ID | 已执行 |
| 3 | 用概念名或QID查找文档和图片；缺QID的Wiki页面按语言＋页面ID查找 | 已执行 |
| 4 | 读取文档全文；核对图片哈希和解码；按请求汇集记录 | 已执行，150条材料记录 |
| 5 | 冻结并保存原始材料包 | 已执行 |
| 6 | 清除明确页面外壳；保留结构、链接与原文块定位 | 已执行，130条文档记录 |
| 7 | 判定概念歧义；筛选可关联文档与图片元数据 | 已实现，本批未调用模型 |
| 8 | 去重文档／图片；按预算选取和截取正文；准备图片字节 | 已实现，本批未执行 |
| 9 | 从所选多篇正文提取带引文的知识候选 | 已实现，本批未调用模型 |
| 10 | 对照同批原文修订候选；记录未解决冲突 | 已实现，本批未调用模型 |
| 11 | 逐图逐知识判定支持区域、范围及局限 | 已接线，本批未执行，真实图像分支待验收 |
| 12 | 保存知识候选、图片支持关系和阻塞记录 | 已实现，本批未执行 |
| 13 | 读取已保存步骤输出与请求 | 可直接查看 |

**名称对应实际代码动作。** 中文标题描述处理内容，末尾英文是目前调用的Python类名；一个类实现多个动作时逐项列出，不把它们笼统称为“整理”。代码名暂未重命名。

下文“本批”指`operator_debug_clean_v2`，旧pilot结果不计入本批。`rows`是一批记录，`bundle`是一个概念的原始材料包。每阶段保留上游数据并增加结果。`limit=100`只限制展示，`SCAN_ROWS`等限制实际处理。


### 配置cell：各算子共同使用的输入参数

| 字段 | 含义 |
| --- | --- |
| `PROJECT / DATASET` | 项目绝对路径／采集数据集根目录。 |
| `MODE` | view_saved只读取旧输出；run执行真实算子。只读模式不把当前配置当成当时运行配置。 |
| `RUN_NAME / RUN` | 运行名称及state/curation下的结果目录；输入、代码或处理配置变化时使用新名称。 |
| `REQUESTS` | 具体概念名／QID列表；这是定向工程输入，不是随机概念采样。 |
| `SOURCE_PATHS` | 本轮选定的采集文件路径；没有选中的文件不会被扫描。 |
| `SCAN_ROWS / IMAGE_BUDGET` | 每源扫描记录上限／每概念在读取阶段检查图片的数量上限。 |
| `RUN_MODEL_CELLS` | 执行模式下是否运行后半段cell；false默认停在清洗。它是notebook执行开关，不是模型生成参数。 |
| `MODEL_CONFIG.base_url / model` | 本地模型服务地址及预期模型名；当前本机8000、qwen3.8-27b。 |
| `MODEL_CONFIG.max_calls` | 一个运行目录允许保留的模型请求数量上限，包含不完整请求；本批配置为12。 |
| `MODEL_CONFIG.max_output_tokens / timeout_s` | 单次最大输出token数／HTTP超时时间秒数。 |
| `MODEL_CONFIG.max_cases` | 命令行pipeline的案例数量上限；当前notebook逐步适配层没有单独执行此校验，应通过REQUESTS控制。 |
| `MODEL_CONFIG.identity_docs / max_docs` | 身份预览文档上限／本轮知识提取选文上限，分别作用于不同阶段。 |
| `MODEL_CONFIG.max_chars_per_doc / max_input_chars` | 单篇正文与合计正文的字符上限，不是整个HTTP请求的token上限。 |
| `MODEL_CONFIG.max_images / max_image_bytes / cos_base_url` | 后续图片数量上限、下载字节上限、可选远程根地址。与读取阶段IMAGE_BUDGET分别设置。 |
| `CONFIG` | 将请求、扫描与图片预算、来源路径和模型配置打包，供冻结和版本匹配。 |
| `session` | 执行模式为DebugSession，经demiflow运行算子；只读模式为SavedDebugSession，仅加载已有结果。 |

模型调用还统一使用固定提示词、temperature=0、JSON对象输出和关闭thinking的服务参数，绑定完整请求保存。具体发送字段在各算子输入表中单列。


In [ ]:
from pathlib import Path
import sys
PROJECT = Path('/yzp/zhaozy/yangzepeng/0905/demiwtg')
if str(PROJECT) not in sys.path: sys.path.insert(0, str(PROJECT))
# notebook 内核可能保留更新前的展示模块；刷新后再导入新增成员。
import importlib
import curation.v4.notebook_debug as notebook_debug
importlib.reload(notebook_debug)
from curation.v4.notebook_debug import DebugSession, SavedDebugSession, show, detail, materials
from curation.v4.sources import discover
from curation.v4.operators import InspectSource, ScanSource, AssembleMaterial, SaveBundle
from curation.v4.contracts import IdentityRegistry, digest, run_lock
from curation.v4.pipeline import DEFAULT
from curation.v4.knowledge_stages import (
    CleanMaterials, ResolveIdentity, OrganizeMaterials, ExtractKnowledge,
    ConsolidateKnowledge, CheckImageSupport, ExportCandidates)
DATASET = PROJECT / 'datasets/demiwtg'
MODE = 'view_saved'  # 只读已有结果；实际调试改为 run，并设置新的 RUN_NAME
RUN_NAME = 'operator_debug_clean_v2'
RUN = PROJECT / 'state/curation/v4' / RUN_NAME
REQUESTS = [{'kind':'legacy','value':'木兰'}, {'kind':'legacy','value':'芦笙'}, {'kind':'qid','value':'Q1'}]
SCAN_ROWS = 50_000  # 每源实际扫描上限；不等于显示条数
IMAGE_BUDGET = 2
MODEL_CONFIG = {**DEFAULT, 'max_cases':3, 'max_calls':12}
# 打开后，执行后半段 cell 会调用本地 Qwen；累计请求上限见 MODEL_CONFIG。
RUN_MODEL_CELLS = False
SOURCE_PATHS = [DATASET / x for x in [
    'meta/concepts.json', 'meta/qid_concepts.fat.jsonl.gz',
    'meta/docs.jsonl', 'meta/images.jsonl', 'corpus/pages-en-part1.jsonl.gz']]
CONFIG = {'requests':REQUESTS, 'scan_rows':SCAN_ROWS, 'image_budget':IMAGE_BUDGET,
          'sources':[str(p) for p in SOURCE_PATHS], 'model':MODEL_CONFIG}
assert MODE in {'view_saved','run'}
session = (SavedDebugSession(RUN) if MODE == 'view_saved' else
           DebugSession(RUN, PROJECT, DATASET, CONFIG, PROJECT/'curation/v4/knowledge_debug.ipynb'))
print('运行目录:', RUN)
print('实际解释器:', sys.executable)

### 完整字段展示

表格中的长字段可以点击“展开全文”，完整数据已保留在输出中。使用 `show(rows, limit=10, chars=None)` 直接显示全文；`detail(row)` 按字段展开，正文保留换行。

默认 `MODE='view_saved'`：重新执行只读取已保存结果，刷新展示，不扫描、不登记身份、不调用模型。实际调试时改为 `MODE='run'`，代码或配置变化请使用新的 RUN_NAME。

## 1．检查来源文件是否可读 · InspectSource

**任务：先确认“有哪些文件可以读”。** 像查收材料目录，还没有阅读所有文章。

- **输入**：`sources`，由 `discover(DATASET, PROJECT)`发现后按`SOURCE_PATHS`选出的5个文件路径和行号，含类型、路径、文件大小及快照信息。
- **实际操作**：检查文件存在性，读取样本记录，查看字段是否可解析。不下载网页、不调用模型。
- **输出**：`inventory_rows`，每源一条，包含`status`、`sample_fields`及读取错误。完整结果保存在`debug_steps/inspect_sources.json`。
- **交给下一步**：扫描仍使用`sources`。当前代码不会根据这个报告自动筛掉某个来源；报告供检查，扫描算子自己处理缺失或错误。

**本批实际**：5个来源分别是旧概念清单、QID概念清单、旧文档清单、图片清单、英文Wiki语料。没有旧`clean_docs`。

**怎么看**：先看路径和类型，再看状态与字段。`readable`只表示样本可读，不表示全文干净、来源可靠或整个文件无坏行。

### 本算子的输入字段

| 输入字段／参数 | 含义 | 从哪里来 |
| --- | --- | --- |
| `sources[].kind` | 来源类型，决定如何解析和匹配。 | discover按已支持的文件路径赋值 |
| `sources[].path` | 要读取的文件绝对路径。 | DATASET及SOURCE_PATHS |
| `sources[].exists` | 文件是否存在。 | 发现来源时检查文件系统 |
| `sources[].size` | 文件字节大小，不是记录数。 | 文件快照 |
| `sources[].mtime_ns / inode` | 文件修改时间（纳秒）和文件系统编号，用于检查运行时文件是否变化；不是内容哈希。 | 文件快照；不存在的文件没有这些字段 |

输入是文件信息列表，不包含文章正文。`InspectSource`每次处理列表中的一个文件。

In [ ]:
sources = [s for s in discover(DATASET, PROJECT) if Path(s['path']) in SOURCE_PATHS]
assert len(sources) == len(SOURCE_PATHS)
inventory_rows = await session.step('inspect_sources', sources, InspectSource)

### 本表字段：来源文件检查

| 字段 | 含义 |
| --- | --- |
| `kind` | 文件类型：legacy_concepts=旧概念；qid_concepts=QID概念；legacy_docs=文档清单；legacy_images=图片清单；wiki_pages=Wiki页面语料。 |
| `path` | 来源文件的绝对路径。 |
| `status` | readable=样本可解析；empty_or_invalid=没有读到有效样本；missing=文件不存在；read_error=读取失败。 |
| `sample_fields` | 样本记录中的字段名称列表；不表示每一行都具备这些字段。 |

表格最左侧`#`只是本次展示顺序，从0开始。抽样后它不等于源文件行号；定位原始数据看provenance。

In [ ]:
show(inventory_rows, limit=100, columns=['kind','path','status','sample_fields'])

## 2．按名称读取概念信息，按QID读取语言页面ID · ScanSource

**从两个文件中查找木兰、芦笙和Q1，保留查到的原始内容。**

| 读取的文件 | 用什么查找 | 读出什么 |
| --- | --- | --- |
| `datasets/demiwtg/meta/concepts.json` | `name`等于“木兰”或“芦笙” | 名称`name`、别名`aliases`、载体`carriers`、分类路径`taxonomy` |
| `datasets/demiwtg/meta/qid_concepts.fat.jsonl.gz` | `qid`等于“Q1” | QID、`en`／`zh`中的页面标题和`page_id`，以及该行已有的其他字段 |

**输入**：`REQUESTS`给出要查的概念名或QID；`sources`给出这两个文件；`SCAN_ROWS`限制每个文件最多扫描多少条。

**输出**：`identity_scans`。`matches`中保留查到的完整内容：`request`说明为哪个请求查到，`record`保存文件中的原始内容，`provenance`保存文件路径、记录位置和哈希。`scan`说明实际查了多少条、是否查到文件末尾。

**本批查到什么**：concepts.json中查到木兰、芦笙；QID文件中查到Q1。其中Q1的`en.page_id=31880`，`en.title=Universe`，第3步可据此查找英文Wiki正文。两个文件均只扫描前50,000条。

**与下一步的联系**：语言＋页面ID帮助第3步查找缺少QID的Wiki页面；名称、别名和taxonomy等保留下来，供第7步判断是否混入同名概念。旧文档和图片仍直接按请求中的概念名查找，目前不使用别名扩展搜索。

**这里没有完成身份核验。** 比如木兰的别名同时含人物与植物名称，本步会照原文件读出来，不能因为读到了就认为这些名称指向同一对象。

### 本算子的输入字段

| 输入字段／参数 | 含义 | 从哪里来 |
| --- | --- | --- |
| `sources（筛选后）` | 只取legacy_concepts、qid_concepts或qid_concepts_base类型；本批实际选了前两种。 | 第1步的来源列表 |
| `REQUESTS[].kind` | legacy表示按旧概念名查找；qid表示按QID查找。 | 配置cell |
| `REQUESTS[].value` | 具体查找值，本批为木兰、芦笙、Q1。 | 配置cell |
| `SCAN_ROWS` | 每个来源最多扫描的记录数，本批50,000；不是抽样数量或显示条数。 | 配置cell |
| `RUN` | 扫描缓存及结果写入位置；不写入datasets。 | 配置cell |
| `session.code` | 绑定执行代码版本的哈希，参与缓存匹配。 | DebugSession冻结当前代码；只读模式取历史manifest |
| `links={}（第三个构造参数）` | 此时不需要页面映射，传入空字典。 | 本cell明确设定 |

此步逐源读取文件中的记录，输出完整匹配内容。请求中的value不在该扫描范围出现时，matches可能为空。

In [ ]:
identity_kinds = {'legacy_concepts','qid_concepts','qid_concepts_base'}
identity_scans = await session.step('scan_identity_records',
    [s for s in sources if s['kind'] in identity_kinds],
    lambda: ScanSource(RUN, REQUESTS, {}, SCAN_ROWS, session.code))

### 本表字段：概念文件扫描

| 字段 | 含义 |
| --- | --- |
| `source` | 这里显示来源类型，不是URL。 |
| `status` | scanned=扫描到了文件末尾；budget_limited=达到扫描预算；missing=文件不存在；read_error=读取出错。 |
| `scan` | 扫描数量、覆盖范围和解析失败记录。 |
| `matches` | 为请求概念找到的原始内容列表；空列表只表示本轮范围未匹配到。 |
| `scan.rows_scanned` | 实际扫描的记录数，不是匹配数。 |
| `scan.complete` | true表示扫描到了该文件末尾；false表示扫描不完整，通常达到预算。 |
| `scan.invalid_rows` | 解析失败的行及原因；空列表表示本轮扫描未记录解析错误。 |
| `request.kind / request.value` | 查找方式与值，例如legacy／芦笙、qid／Q1。 |
| `record` | 该来源文件里匹配到的完整原始内容；字段随文档／图片／概念类型不同，见下方来源字段说明。 |
| `provenance.source_path` | 原始记录来自哪个文件。 |
| `provenance.line` | JSONL／TSV文件中的行号，从1开始。 |
| `provenance.raw_line_sha256` | 该原始行字节的SHA256；用于识别这次读到的版本。 |
| `provenance.record_index` | JSON数组内记录序号，从1开始；不是文本文件行号。 |
| `provenance.source_content_sha256` | 读取JSON整文件时计算的内容哈希。与逐行文件的raw_line_sha256不同。 |
| `association_method` | 关联依据：exact_source_identity表示按原有名称／QID等标识匹配；exact_language_page_id_from_sitelink表示按语言＋页面ID匹配。均不是语义审核结论。 |


In [ ]:
show([{'source':s['source']['kind'], 'status':s['status'], 'scan':s['scan'], 'matches':s['matches']} for s in identity_scans])

## 3．按概念名／QID／页面ID查找文档和图片 · ScanSource

**任务：根据概念名、QID和刚读到的语言页面ID，收集“可能属于这个概念”的文档和图片记录。** 此时只确定原有清单如何关联，不裁决材料是否正确。

- **输入**：`identity_scans`、`REQUESTS`、3个材料来源与扫描上限。
- **实际操作**：先从QID记录构造`links`，即QID到`(语言, page_id)`的映射。旧文档／图片按记录已有的概念标签匹配；Wiki优先按QID匹配，没有QID时按明确的语言＋页面ID对应。不会仅凭同名猜QID。
- **输出**：`material_scans`，结构同上一步：匹配记录`matches`、行号／哈希等`provenance`、关联依据及扫描覆盖范围。
- **交给下一步**：与`identity_scans`合成`scans`，用于按概念组装材料包。

**本批实际**：文档来源完整扫描10,023行，命中129条；图片来源扫描前50,000行，命中17条；Wiki来源扫描前50,000行，命中1条。这里统计的是记录，不是去重后的独立文章／图片。

**怎么看**：展开一条`match`核对请求、记录和文件路径和行号。木兰的同名异义材料可以在这一步进入，下一步不会因其已被采集标签关联就自动认为正确。显示抽样只抽查命中记录，不改变实际处理材料。

### 本算子的输入字段

| 输入字段／参数 | 含义 | 从哪里来 |
| --- | --- | --- |
| `identity_scans[].source.kind` | 用来只取QID来源的匹配内容构建映射。 | 第2步输出 |
| `identity_scans[].matches[].record.qid` | 映射的概念QID，例如Q1。 | 第2步读到的QID文件内容 |
| `record.en / record.zh` | 语言页面对象；本cell读取其中的page_id。 | 第2步匹配到的原始内容 |
| `links` | QID到(语言,page_id)列表的字典，例如Q1对应(en,31880)；只用于定位缺QID的Wiki页。 | 本cell从上述字段计算并去重 |
| `sources（筛选后）` | 除概念信息来源外的选定文件：本批是文档清单、图片清单、Wiki语料。 | 第1步来源列表 |
| `REQUESTS[].kind` | legacy表示按旧概念名查找；qid表示按QID查找。 | 配置cell |
| `REQUESTS[].value` | 具体查找值，本批为木兰、芦笙、Q1。 | 配置cell |
| `SCAN_ROWS` | 每个来源最多扫描的记录数，本批50,000；不是抽样数量或显示条数。 | 配置cell |
| `RUN` | 扫描缓存及结果写入位置；不写入datasets。 | 配置cell |
| `session.code` | 绑定执行代码版本的哈希，参与缓存匹配。 | DebugSession冻结当前代码；只读模式取历史manifest |

旧文档／图片的名称匹配直接使用REQUESTS，不使用别名扩展；Wiki记录自身有QID时按QID匹配。

In [ ]:
links = {}
for scan in identity_scans:
    if scan['source']['kind'] not in {'qid_concepts','qid_concepts_base'}: continue
    for match in scan['matches']:
        record = match['record']
        for lang in ['en','zh']:
            site = record.get(lang) or {}
            if site.get('page_id') is not None:
                links.setdefault(record['qid'], []).append((lang,site['page_id']))
links = {k:sorted(set(v)) for k,v in links.items()}
material_scans = await session.step('scan_materials',
    [s for s in sources if s['kind'] not in identity_kinds],
    lambda: ScanSource(RUN, REQUESTS, links, SCAN_ROWS, session.code))

### 两张表的字段：文档／图片扫描与命中内容

| 字段 | 含义 |
| --- | --- |
| `source / status` | 与上一步相同：来源类型及扫描状态。 |
| `matched` | 此来源匹配出的记录数，即len(matches)；一条来源可能关联多个请求，不能直接当作独立材料数。 |
| `scan` | 本来源的扫描范围与错误。 |
| `scan.rows_scanned` | 实际扫描的记录数，不是匹配数。 |
| `scan.complete` | true表示扫描到了该文件末尾；false表示扫描不完整，通常达到预算。 |
| `scan.invalid_rows` | 解析失败的行及原因；空列表表示本轮扫描未记录解析错误。 |
| `request.kind / request.value` | 查找方式与值，例如legacy／芦笙、qid／Q1。 |
| `record` | 该来源文件里匹配到的完整原始内容；字段随文档／图片／概念类型不同，见下方来源字段说明。 |
| `provenance.source_path` | 原始记录来自哪个文件。 |
| `provenance.line` | JSONL／TSV文件中的行号，从1开始。 |
| `provenance.raw_line_sha256` | 该原始行字节的SHA256；用于识别这次读到的版本。 |
| `provenance.record_index` | JSON数组内记录序号，从1开始；不是文本文件行号。 |
| `provenance.source_content_sha256` | 读取JSON整文件时计算的内容哈希。与逐行文件的raw_line_sha256不同。 |
| `association_method` | 关联依据：exact_source_identity表示按原有名称／QID等标识匹配；exact_language_page_id_from_sitelink表示按语言＋页面ID匹配。均不是语义审核结论。 |

第一张表每行是一个文件的扫描结果；第二张表每行是一次匹配。第二张表固定种子抽样，但完整命中内容仍保留。

In [ ]:
show([{'source':s['source']['kind'], 'status':s['status'], 'matched':len(s['matches']), 'scan':s['scan']} for s in material_scans])
show([m for s in material_scans for m in s['matches']], limit=100, sample=True, seed=42)

## 4．读取文档正文，核对图片字节，按请求汇集记录 · AssembleMaterial

**任务：把清单中的“材料地址”变成后续算子能够读取的内容，并按概念装包。**

- **输入**：`scans`中的匹配记录；`tasks`中的请求、内部试运行ID与任务ID；数据集路径和图片预算。
- **实际操作**：对旧文档按记录的`path`读取本地页面全文，记录内容哈希或读取失败；Wiki保留原有章节记录。每概念最多检查2张图片的本地字节哈希、解码及尺寸，剩余图片记录仍保留。内部ID只写试运行注册表，不改权威概念清单。
- **输出**：`bundles`，每概念一个材料包。`materials`是具体材料；`document.text`是读到的旧页面正文；`bytes`是图片字节检查；`coverage`与`gaps`记录范围和缺口。
- **交给下一步**：原始材料包先保存，再进入新清洗算子。

**本批实际**：木兰115条（1身份＋114文档）；芦笙33条（1身份＋15文档＋17图片）；Q1共2条（1身份＋1Wiki页面），合计150条。

**怎么看**：在完整记录里查看`document.status`、正文和`bytes.status`。`verified_bytes`只证明文件哈希和解码通过；`not_local`不代表没有相关图片。这里没有清洗正文，也没有看图确认主体。

### 本算子的输入字段

| 输入字段／参数 | 含义 | 从哪里来 |
| --- | --- | --- |
| `tasks[].request` | 本任务要汇集哪个概念的材料，含kind和value。 | REQUESTS |
| `tasks[].concept_id` | 当前独立试运行注册表分配的内部ID，不回写权威概念清单。 | IdentityRegistry.get(request) |
| `tasks[].task_id` | 绑定配置、请求及扫描输出的任务哈希，用于保存材料包。 | 本cell计算 |
| `scans[].source.kind / path` | 材料来自哪个类型和文件。 | 第2、3步输出合并 |
| `scans[].matches[].request` | 判断匹配内容属于当前哪个请求。 | 第2、3步输出 |
| `scans[].matches[].record` | 要汇集的原始内容；旧文档读取record.path，图片读取path和sha256，Wiki直接保留sections等字段。 | 采集文件原始内容 |
| `scans[].matches[].provenance / association_method` | 来源文件位置、哈希及关联方法。 | 扫描算子输出 |
| `scans[].status / scan` | 原来源扫描是否完整及错误，传入材料包coverage。 | 扫描算子输出 |
| `DATASET` | 解析采集记录相对path时使用的根目录。 | 配置cell |
| `blob_roots=[]（第三个构造参数）` | 额外查找本地图片的目录列表；本批为空，仅查DATASET。 | 本cell |
| `IMAGE_BUDGET` | 每概念最多核对几张图片，本批为2；每检查一条就占一个额度，失败也占。 | 配置cell |

只读模式直接加载保存的材料包，tasks设为空且不运行算子，不重新写入身份注册表。

In [ ]:
scans = sorted(identity_scans + material_scans, key=lambda s:s['source']['path'])
if MODE == 'run':
    with run_lock(PROJECT/'state/curation/v4'):
        registry = IdentityRegistry(PROJECT/'state/curation/v4/identities.sqlite')
        try:
            tasks = [{'request':r, 'concept_id':registry.get(r),
                      'task_id':digest({'config':CONFIG,'request':r,'scans':scans})} for r in REQUESTS]
        finally: registry.close()
else:
    tasks = []  # 只读模式不登记身份、不改写运行文件
bundles = await session.step('assemble_material_bundle', tasks,
    lambda: AssembleMaterial(scans, DATASET, [], IMAGE_BUDGET))

### 本表字段：读到的文档与图片

| 字段 | 含义 |
| --- | --- |
| `case` | 材料对应的请求，例如legacy:木兰或qid:Q1。 |
| `kind` | 采集文件类型，决定record内有哪些字段。 |
| `title` | 源记录中的页面标题；图片或概念记录可能没有，因此显示None。 |
| `text` | 文档全文，或Wiki的章节列表；身份信息／图片没有正文时为None。这里展示原始版本。 |
| `bytes` | 图片字节检查对象；非图片通常为None。 |
| `bytes.status` | 图片字节处理状态：verified_bytes=本地哈希与解码通过；not_local=检查路径未找到；budget_limited=超出本轮检查数量；missing_locator=没有路径；invalid_path=路径越界；missing_expected_hash=缺少预期哈希；hash_mismatch=哈希不符；decode_error=无法解码。 |
| `bytes.path / sha256` | 已找到图片的绝对路径及经计算核对的内容哈希。 |
| `bytes.dimensions / format` | 实际解码得到的[宽,高]与图片格式。 |
| `bytes.actual_sha256` | 遇到预期哈希缺失或不匹配时记录的实际哈希。 |
| `bytes.remote_locator` | 源记录中的远程地址，可能不是直接图片字节链接。 |
| `bytes.error / note` | 错误或范围说明。 |
| `bytes.knowledge_support / generation_origin` | 分别表示知识支持尚未审核、是否生成图尚未核实；不能从字节校验通过推断。 |


In [ ]:
raw_materials = materials(bundles)
show(raw_materials, limit=100, sample=False, columns=['case','kind','title','text','bytes'])

### 查看完整单条材料／采样／图片
改索引、limit和seed只执行展示，不重新扫描或调用模型。

### 本表与完整单条内容的字段

| 字段 | 含义 |
| --- | --- |
| `kind / title / text / bytes` | 含义同上一张材料表；这里限定到CASE指定概念。 |
| `record（case_materials[i]中）` | 用于查看完整材料的包装对象，包含kind、record、provenance、document或bytes；它并非只包含来源原始行。 |
| `record（完整材料对象中）` | 内层record才是文件中原始记录的字段，不是模型新生成内容。 |
| `document.path / sha256 / text / status` | 旧页面的绝对文件路径、实际内容哈希、全文和读取状态readable或read_error。 |
| `document.error` | 原始正文读取失败的原因。 |
| `knowledge_support` | 初始not_reviewed：材料还没有通过知识证据审核。 |
| `caption.existing` | 原图片记录已有的caption；当前未确认它是来源图注还是模型生成描述。 |
| `caption.provenance / supplemental` | 现有描述的来源分类说明，以及本步骤尚未补写的描述（null）。 |
| `provenance.source_path` | 原始记录来自哪个文件。 |
| `provenance.line` | JSONL／TSV文件中的行号，从1开始。 |
| `provenance.raw_line_sha256` | 该原始行字节的SHA256；用于识别这次读到的版本。 |
| `provenance.record_index` | JSON数组内记录序号，从1开始；不是文本文件行号。 |
| `provenance.source_content_sha256` | 读取JSON整文件时计算的内容哈希。与逐行文件的raw_line_sha256不同。 |
| `association_method` | 关联依据：exact_source_identity表示按原有名称／QID等标识匹配；exact_language_page_id_from_sitelink表示按语言＋页面ID匹配。均不是语义审核结论。 |

下方detail按字段展示完整对象。`case_materials[i]`的i是筛选后列表索引，与抽样表最左侧#不一定相同。

In [ ]:
CASE = 'legacy:芦笙'
case_materials = [m for m in raw_materials if m['case']==CASE]
show(case_materials, limit=100, sample=True, seed=42, columns=['kind','title','text','bytes'])
if case_materials: detail(case_materials[0]['record'])
# 查看另一条完整输入：detail(case_materials[3]['record'])
# 看已经核验字节的本地图（按需执行）：
# from IPython.display import Image, display
# display(Image(filename=case_materials[索引]['record']['bytes']['path']))

### 展开原始record：本批采集文件中的字段

| 字段 | 含义 |
| --- | --- |
| `name / aliases / carriers / taxonomy` | concepts.json中的名称、别名、载体类型、分类路径快照；并非新审核结果。 |
| `qid / en / zh` | QID及英文／中文页面信息；en和zh内的title为标题，page_id为对应语言站点的页面ID。 |
| `p18 / p373` | QID来源给出的P18图片地址和P373 Commons分类名称；未核验图片支持关系。 |
| `concepts / instances` | 采集清单中已有的关联概念名称。字段名不同来自两类清单，不意味着语义已核验。 |
| `url / title / path（文档）` | 抓取地址、标题、相对于数据集的本地正文路径。 |
| `page_sha` | 采集页面定位标识，旧页面按URL哈希寻址；不能当作正文内容哈希。 |
| `authority` | 采集端的来源路由／类型标签（如wiki、serp），不是本次可靠性评分。 |
| `query / fetched_at` | 采集时使用的查询词／源记录的抓取时间值；原样保留。 |
| `n_images / n_passages` | 采集端报告的图片数／片段数；不等于本轮实际获取或输入模型的数量。 |
| `sha256 / path / ext / mime / size_bytes（图片）` | 源记录的图片内容哈希、数据集内路径、扩展名、MIME类型和文件字节数；需与bytes实际核验区分。 |
| `width / height / orig_width / orig_height` | 来源记录的宽高及原始宽高字段；本轮实际解码尺寸看bytes.dimensions。 |
| `content_url / landing_url` | 图片内容地址／图片所在网页地址；不保证当前仍可访问。 |
| `author / license / source` | 采集端记录的作者、许可和来源；未在本轮独立核验。 |
| `caption` | 源记录已有描述，可能是历史模型生成；不自动当作原始图注或像素事实。 |
| `queries / query_langs` | 图片采集使用的查询词及语言。 |
| `identity / focus / quality / kb_match / richness` | 图片记录携带的历史判断或打分字段；本流程原样保留，未重新计算，也未将其当当前审核依据。具体取值尺度依该记录生成版本，不从名称猜测分数含义。 |
| `lang / page_id / revision_id` | Wiki语言、页面ID和页面修订版本；page_id必须结合语言使用。 |
| `sections` | Wiki章节内容；每项保留标题title和正文text等原有字段。 |
| `is_disambig / is_redirect / redirect_target` | Wiki来源标记的消歧义页、重定向页及重定向目标。 |
| `links / link_count / images / categories（Wiki）` | Wiki来源解析得到的链接、链接数、图片及分类信息；不等于已下载图片数或本项目taxonomy。 |
| `parser_version / text_sha256 / byte_len` | Wiki采集解析器版本、采集端报告的文本哈希和字节长度；原样保留，不能替代本次cleaning.source_sha256。 |

该表按本批实际出现的字段编写。未经本轮核验的历史字段只解释来源与用途，不把其数值当作当前质量结论。

## 5．冻结并保存原始材料包 · SaveBundle

**任务：把本轮实际拿到的原始材料固定下来，后面的调试都能回到同一份输入。**

- **输入**：第4步的`bundles`。
- **实际操作**：将每个材料包保存到运行目录的`bundles/<task_id>.json`，拒绝以不同内容覆盖同一路径。通过notebook适配层还会保存该步骤的完整输出。
- **输出**：`saved_bundles`，内容仍是原始材料包，没有新增知识、清洗结论或审核标签。
- **交给下一步**：每包包装成`{case_id, bundle}`，作为清洗及后续知识阶段的一条输入。

**本批实际**：保存3个材料包。相同输入重复执行会读取已保存结果，不重复读取页面、验图或调用模型。

**怎么看**：检查每包请求、材料数量和覆盖范围。原始材料都固定了，不等于已经拿齐整个概念的材料。

### 本算子的输入字段

| 输入字段／参数 | 含义 | 从哪里来 |
| --- | --- | --- |
| `bundles[].task_id` | 决定保存文件名bundles/<task_id>.json。 | 第4步输出 |
| `bundles[].concept_id / request` | 本包所属内部ID及原始查找请求。 | 第4步输出 |
| `bundles[].materials` | 完整原始材料列表，可能包含身份信息、文档、图片。 | 第4步输出 |
| `bundles[].coverage / gaps` | 扫描范围及明确缺口。 | 第4步输出 |
| `bundles[].schema / stage / knowledge_status / identity_status` | 材料包结构版本、生成阶段、尚未提取知识及仅按已有来源标识关联的状态说明。 | 第4步输出 |
| `RUN` | 保存原始材料包的运行目录。 | 配置cell |

整个bundle原样保存；此步不修改文档文本或图片，也不生成知识。

### 本表字段：已保存材料包

| 字段 | 含义 |
| --- | --- |
| `request` | 本包对应的{kind,value}请求。 |
| `materials` | 这里是材料条数len(b["materials"])；原始bundle中的同名字段则是完整材料列表。 |
| `coverage` | 各相关文件的扫描范围列表。每项source_path是文件，kind是类型，status是扫描状态，scan是扫描计量；含义同第2、3步。 |


In [ ]:
saved_bundles = await session.step('save_material_bundle', bundles, lambda: SaveBundle(RUN))
show([{'request':b['request'],'materials':len(b['materials']),'coverage':b['coverage']} for b in saved_bundles])

## 6．清除明确页面外壳，保留内容块及原文定位 · CleanMaterials

**任务：减少页面外壳对阅读的干扰，同时保留能够回查的证据。** 例如把整行“登录”从阅读正文中排除，但不能删掉“这里讨论登录协议”的正文。

- **输入**：`knowledge_input`，每条包含`case_id`与完整原始`bundle`；只处理其中的采集文档／Wiki页面，概念信息和图片记录继续保留。
- **实际操作**：普通HTML按结构块处理；Markdown、文本、Wiki片段按行保留结构。排除明确整行导航、空行和少量布局模板；保留短定义、表格、引用、图注和图片链接。未知Wiki模板、混合HTML、登录墙线索标为待审。相同清洗正文只记重复关联，不删除原始材料。
- **输出**：`cleaned_rows`。其中`bundle`保持原样，`cleaned_materials`保存带`cleaning`的材料，`cleaning_summary`方便概览。
- **交给下一步**：identity读取`cleaned_materials`中的清洗文本；不会再把历史clean_docs当输入。

| 清洗字段 | 含义与查看方法 |
| --- | --- |
| `text` | 供后续阅读的清洗正文；不能当作原始下载文件原样全文 |
| `blocks` | 内容块及处理决定；展开可查为什么保留或排除 |
| `raw_start/raw_end/raw_text` | 在被冻结源文本中的范围与原始片段，范围右端不包含 |
| `clean_start/clean_end` | 保留块在清洗正文中的范围；排除块没有这个范围 |
| `links/images` | 从块中保留的链接、图片地址和文字线索；不证明图片实际可见内容 |
| `status/warnings` | 机械处理状态与需审问题，不是知识可靠性结论 |

**本批实际**：130条文档记录完成处理；114条标为`cleaned_candidate`，16条`needs_review`（木兰14、芦笙1、Q1为1）。全部原文块及清洗块偏移均已核对。这里的记录数不是独立来源数。

**怎么看**：先看前后对照，再抽查排除块与仍保留的噪声。`needs_review`目前是提示，**不会自动阻止非空文本进入identity**；因此本轮先在此人工查看，不应把所有清洗候选当合格材料。

**实现范围**：复杂模板和部分导航仍残留。映射为原始内容块，不是加工文字到原始字节的一一对应；Wiki偏移针对明确的章节序列化文本。清洗不纠正事实、不判断木兰是哪一个概念，也不提取知识。

### 本算子的输入字段

| 输入字段／参数 | 含义 | 从哪里来 |
| --- | --- | --- |
| `knowledge_input[].case_id` | 绑定这份原始材料包的案例哈希，用于逐案例保存阶段文件。 | 本cell从concept_id及bundle计算 |
| `knowledge_input[].bundle` | 冻结的完整原始材料包。 | 第5步saved_bundles |
| `bundle.materials[].kind` | 决定是否处理文字：legacy_docs、wiki_pages进入清洗；其他类型保留不变。 | 采集来源类型 |
| `materials[].document.text` | 旧页面的本地全文；未读到时为空。 | 第4步实际读取文件 |
| `materials[].record.sections[].title / text` | Wiki各章节标题及正文；本算子明确串联成文本后定位。 | 采集语料中的Wiki页面 |
| `materials[].provenance / document.path / document.sha256` | 源记录位置、本地文件路径与读到的内容哈希，绑定清洗来源。 | 扫描及读文件步骤 |
| `materials[].record` | 完整原始内容，不被清洗文本覆盖。 | 采集文件 |
| `清洗规则版本VERSION` | 当前conservative-blocks/1；控制保守外壳排除与内容块解析。 | cleaning.py；修改代码需另开run |

这个算子不向模型发送任何输入。MODEL_CONFIG虽由通用阶段适配层传入，清洗本身不消耗模型调用额度。

In [ ]:
knowledge_input = [{'case_id':digest({'concept_id':b['concept_id'],'bundle':b})[:20],
                    'bundle':b} for b in saved_bundles]
cleaned_rows = await session.knowledge(CleanMaterials, knowledge_input)
identity_rows = organized_rows = extracted_rows = consolidated_rows = evidence_rows = exported_rows = []

### 两张清洗概览表的字段

| 字段 | 含义 |
| --- | --- |
| `case` | 第一张表为{kind,value}；第二张表仅显示概念名／QID。 |
| `index（第一张表）` | 该材料在当前概念cleaned_materials列表中的索引，从0开始，包含身份及图片造成的索引间隔。 |
| `index（第二张表）` | 该文档在跨概念clean_material_rows列表中的索引，从0开始；修改CLEAN_INDEX使用这个值。 |
| `kind` | 第一张表中的原材料类型，legacy_docs或wiki_pages。 |
| `title` | 原文档标题。 |
| `status` | cleaned_candidate=机械清洗候选；needs_review=有提示问题或正文为空；unavailable=未拿到源文本。都不是事实核验状态。 |
| `counts` | 字符数、块数与排除数统计。 |
| `warnings` | 第一张表中的问题代码列表；空列表不保证无噪声。 |
| `counts.input_chars / output_chars` | 输入源文本／清洗文本的字符数；不是字节数或token数。 |
| `counts.blocks` | 本次解析得到的内容块数量。 |
| `counts.excluded` | 从阅读正文排除的块数，包含空行；不能直接当成删除噪声的有效数量。 |
| `warnings.no_body_blocks` | 没有识别出正文类型的非空内容块。 |
| `warnings.access_restriction_or_challenge_marker` | 存在登录限制／访问拒绝／验证码等线索；可能未拿到正文。 |
| `warnings.unexpanded_wiki_template_retained` | 保留了尚未展开的Wiki模板。 |
| `warnings.mixed_html_markup_retained` | 非完整HTML输入中仍有HTML片段。 |
| `warnings.malformed_html_tail_retained` | HTML尾部结构不完整，原始尾部保留供检查。 |
| `warnings.auxiliary_html_retained` | 保留了header／aside等可能属于辅助内容的HTML块。 |

第二张表的index可以直接复制给下一cell的CLEAN_INDEX；最左侧#始终只是本次展示顺序。

In [ ]:
show([{'case':r['bundle']['request'], **s} for r in cleaned_rows for s in r['cleaning_summary']], limit=100)
clean_material_rows = [{'case':r['bundle']['request']['value'], 'material':m}
    for r in cleaned_rows for m in r['cleaned_materials'] if 'cleaning' in m]
show([{'index':i,'case':r['case'],'title':r['material']['record'].get('title'),
       'status':r['material']['cleaning']['status'], 'counts':r['material']['cleaning']['counts']}
      for i,r in enumerate(clean_material_rows)], limit=100)

### 前后对照及内容块表字段

| 字段 | 含义 |
| --- | --- |
| `title / original / cleaned / warnings` | 前后对照中的原标题、源全文、清洗全文和问题代码。 |
| `block_id` | 由块的源起止位置与原文生成的标识；只用于引用该块，不能作跨文档永久ID。 |
| `kind` | 块类型，例如heading标题、paragraph段落、list列表、table表格、figure图、quality_note来源质量提示、template模板；HTML还可能保留p等标签名。 |
| `raw_start / raw_end` | 在源文本中的字符范围，0开始、左闭右开。Wiki源文本采用已说明的章节串联方式。 |
| `raw_text` | 源文本在该范围内的完整内容，含原始标记／换行。 |
| `text` | 块的可读文字版本；即使该块被排除也保存，方便核查删除依据。 |
| `decision` | keep=保留；exclude=不拼入阅读正文。原文仍在raw_text中。 |
| `reason` | retained=保留；blank=空白；exact_navigation_line=匹配明确导航整行；layout_template=布局模板；html_nav／html_footer／html_form／html_script／html_style=对应HTML外壳。 |
| `clean_start / clean_end` | 展开完整块时可见：该块在cleaning.text中的字符范围；被排除或没有可读文字的块无此字段。 |
| `section` | 最近识别的标题文本列表；当前并不是完整多级章节树。 |
| `links / images` | 保留的链接和图片线索。target为地址或Wiki目标，label为原有文字，alt为HTML图片替代文字（如有），kind为提取来源类型。 |
| `alignment` | whole_source_block：映射到整块源文本；不保证每个清洗字符都有一一对应的源字符。 |

只想审删除情况，可用下面已提供的注释代码筛选decision==exclude。

In [ ]:
# 可交互修改 CLEAN_INDEX；这里只查看，不重跑清洗。
CLEAN_INDEX = next((i for i,r in enumerate(clean_material_rows)
    if r['material']['record'].get('title') == '《花木兰》明斯克外访 - 香港舞蹈團'), 0)
if clean_material_rows:
    chosen = clean_material_rows[CLEAN_INDEX]['material']
    from curation.v4.cleaning import raw_text
    detail({'title':chosen['record'].get('title'), 'original':raw_text(chosen),
            'cleaned':chosen['cleaning']['text'], 'warnings':chosen['cleaning']['warnings']})
    show(chosen['cleaning']['blocks'], limit=100, sample=False,
         columns=['block_id','kind','raw_start','raw_end','raw_text','text','decision','reason'])
    # 可单独检查排除块：
    # show([b for b in chosen['cleaning']['blocks'] if b['decision']=='exclude'], limit=100)

## 7．判定概念歧义，筛选可关联文档与图片元数据 · ResolveIdentity

**任务：判断本轮请求具体指向什么，以及哪些材料可继续参与处理。** 与第2步不同，这里才调用模型做语义判断。

- **输入到算子**：`cleaned_rows`。**实际给模型**：请求、概念信息、最多12篇选中文档的前500字符，以及前2张图的标题／URL／caption等元数据；不传图片像素，也不传全文。
- **实际操作**：比较身份线索；概念本身有未解歧义则阻塞；为已预览文档及图片元数据逐一给出接受或排除。程序检查返回格式与材料ID覆盖。
- **输出**：`identity_rows`，新增`identity`（状态、目标名、接受／排除及理由）、`identity_materials`（实际预览的完整材料记录）、`identity_unexamined`（未查材料ID），必要时新增`blocked`。
- **交给下一步**：organize只拿预览过且接受的材料。被遗漏于预览的材料当前不会再由organize主动探索。

**本批状态：尚未调用模型。** 先前旧批的木兰歧义只能作为调试经验，不能冒充本次清洗输入上的新结果。

**怎么看**：核对判断是否真有输入依据，是否把相关主题误当同一身份，是否仅凭caption声称看见图片。`resolved`仅是本次有限预览上的机器判断，既不完成跨源永久合并，也不证明知识正确。当前接受／拒绝二分仍会混淆“暂缓”与“错配”，需继续review。

**“过滤概念无关文档”的边界**：此处模型筛选的是材料与目标概念的关联，不应把“页面主题不是目标本身”直接等同于无关。例如介绍芦笙舞的文档仍可能提供芦笙使用情境。图片在此只判断元数据线索，像素支持留到第11步。

### 本算子的输入字段

| 输入字段／参数 | 含义 | 从哪里来 |
| --- | --- | --- |
| `cleaned_rows[].case_id / bundle.request` | 案例标识与要判断的概念请求。 | 第6步保留的输入字段 |
| `cleaned_rows[].cleaned_materials` | 清洗后的文档，以及继续保留的概念信息、图片元数据。 | 第6步输出 |
| `materials[].record（概念信息）` | 名称、别名、taxonomy，或QID及语言页面对应关系。 | 第2步原始内容，经材料包与清洗步骤保留 |
| `materials[].cleaning.text` | 用于选取身份预览的清洗正文。 | 第6步 |
| `materials[].record.title / url / content_url / caption` | 标题、页面／图片地址和已有描述线索，可能缺失。 | 原始采集内容 |
| `MODEL_CONFIG.identity_docs` | 最多预览多少个页面组，本批12。 | 配置cell，默认配置合入 |
| `MODEL_CONFIG.max_images` | 最多预览多少条图片元数据，本批2。 | 配置cell |
| `每文档500字符` | 所选清洗文本只取前500字符；是当前代码常量，不是全文已审。 | ResolveIdentity实现 |

**模型实际输入**只有`request`、`source_identity_records`和`materials`预览列表；预览每项包含`material_id/kind/title/url/caption/text_preview`。`material_id`供模型引用，其他字段含义如上。原始全文、清洗块、图片像素及未选材料不发送。

In [ ]:
identity_rows = []
if (MODE == 'view_saved' or RUN_MODEL_CELLS) and cleaned_rows:
    identity_rows = await session.knowledge(ResolveIdentity, cleaned_rows)
else:
    print('未执行：模型调试开关关闭，或前一步没有输出。')

### 本表字段：概念歧义与材料关联判断

| 字段 | 含义 |
| --- | --- |
| `case` | 该结果对应的概念请求；常为{kind,value}，材料表中显示legacy:芦笙这样的字符串。 |
| `blocked` | 本案例不能继续的原因；null表示未记录阻塞。对象内stage是发生阶段，reason是类别，detail是可选详细说明。 |
| `result` | 当前这一步的结果对象，可点击展开；具体字段见本表后续各行。 |
| `result.status` | resolved=本次预览可确定目标；ambiguous=目标存在歧义；insufficient=依据不足。后两者阻塞。 |
| `result.target_label` | 模型给出的目标概念展示名称，不修改权威名称。 |
| `result.reason` | 目标身份判断的理由或缺口。 |
| `result.accepted_material_ids` | 本次接受的材料ID列表，指向identity_materials。 |
| `result.rejected_materials` | 排除列表；每项material_id为材料标识，reason为理由。当前也可能混入因歧义暂缓的材料。 |
| `result.identity_groups` | 模型列出的不同身份分组说明，不能当永久合并／拆分结果。 |
| `result.reviewer` | 说明结果来自本地模型，不是人工裁定。 |
| `call.request_path / response_path` | 完整模型请求／原始响应的保存路径。 |
| `call.model` | 响应记录中的模型名称。 |
| `call.elapsed_s` | 调用耗时，单位秒。 |
| `call.usage` | 服务返回的token计量，如prompt_tokens、completion_tokens、total_tokens；不是知识质量分。 |

本批还没有模型输出，因此表为空。展开其他字段时，identity_materials是实际预览材料，identity_unexamined是未查看材料的ID。

In [ ]:
show([{'case':r['bundle']['request'],'blocked':r.get('blocked'),'result':r.get('identity')} for r in identity_rows], limit=100)
if identity_rows: detail(identity_rows[0].get('identity', identity_rows[0].get('blocked')))

## 8．去重文档与图片，按预算选取并截取正文，准备图片字节 · OrganizeMaterials

**这一步的具体产物是：本次模型将阅读的正文片段列表，以及能够传给图像模型的图片文件列表。** 目前类名虽叫OrganizeMaterials，代码实际执行下面几个动作。

**输入**：`identity_rows`中的接受材料ID、对应清洗文档、图片记录及预算。身份已阻塞的案例跳过本步。

| 顺序 | 动作与处理对象 | 实际规则 | 产出／记录 |
| --- | --- | --- | --- |
| 1 | 取出身份阶段已接受的材料 | 只取`identity_materials`中ID在接受列表里的记录 | 待选文档与图片；不重新判断相关性 |
| 2 | 去重正文与图片 | 正文将空白归一后比较是否完全相等；图片比较SHA256 | 保留代表材料，在`duplicates`中记录重复关系 |
| 3 | 按页面地址分组选取正文 | 当前按材料ID排序，每个页面组取一个版本，再取最多2篇 | 本轮选中文档；其他记录写入`omissions` |
| 4 | 按字符预算截取正文 | 每篇从开头取最多6,500字符，合计最多13,000字符 | `passages`：片段正文、来源ID、起止位置和原文块映射；截去的尾部记入`omissions` |
| 5 | 准备图片字节 | 对预算内已接受图片复验本地哈希，或调用已配置的获取适配 | 可用者进入`images`，失败者进入`image_gaps` |
| 6 | 检查是否有可提取正文 | 没有选出任何正文片段则阻塞 | `blocked`；当前尚不支持仅图片知识提取 |

**输出**：`organized_rows`新增`material_pack`，其中包含`passages`、`images`、`duplicates`、`omissions`和`image_gaps`。第9步读取passages，第11步使用images。

**举例说明规则（不是本批新结果）**：如果上一步接受5篇不同页面的文档，其中有完全相同的正文，先记重复关系；剩余文档按当前顺序最多取2篇。如果一篇有20,000字符，本次只送前6,500字符，尾部明确记为未读。其余页面还保留在上游记录中。

**本批状态：未执行。** 当前不是按权威性选最佳文章，也不根据知识缺口选章节；它只是按既定机械顺序与预算取材料。更不会在这里再次过滤概念无关文档——相关性筛选发生在第7步。

**调试时看三件事**：选中哪几篇、丢在预算外的是什么、截取是否丢掉关键条件。材料ID排序和只取开头是待改进实现，不是研究约束。已保留来源映射不等于所选材料足够可靠或完整。


### 本算子的输入字段

| 输入字段／参数 | 含义 | 从哪里来 |
| --- | --- | --- |
| `identity_rows[].blocked` | 前一步阻塞则跳过，不继续选取。 | 第7步或更早阶段 |
| `identity_rows[].identity.accepted_material_ids` | 本次允许继续处理的材料ID。 | 第7步模型输出 |
| `identity_rows[].identity_materials` | 实际预览过的完整材料；不会从未预览列表自动补选。 | 第7步选择 |
| `materials[].material_id / record.url` | 材料标识与页面地址，参与排序和页面分组。 | 第7步／采集内容 |
| `materials[].cleaning.text / blocks / version / source_sha256 / source_locator` | 清洗全文、原文映射、清洗版本与来源信息；供去重、截取和留存证据。 | 第6步 |
| `materials[].record.path / sha256 / bytes` | 图片原始定位和预期哈希，以及前面字节核验结果。 | 采集清单与第4步 |
| `MODEL_CONFIG.max_docs / max_chars_per_doc / max_input_chars` | 最多2篇、每篇6,500字符、总13,000字符，均为本轮选取预算。 | 配置cell |
| `MODEL_CONFIG.max_images / cos_base_url / max_image_bytes` | 图片数量上限、可选远程根地址、单次下载字节上限；当前远程根地址为空。 | 配置cell |

此步不调用文字模型。准备好的正文映射留在输出中，不能借此把未选原文送入下一步模型。

In [ ]:
organized_rows = []
if (MODE == 'view_saved' or RUN_MODEL_CELLS) and identity_rows:
    organized_rows = await session.knowledge(OrganizeMaterials, identity_rows)
else:
    print('未执行：模型调试开关关闭，或前一步没有输出。')

### 本表字段：选中正文片段与可用图片

| 字段 | 含义 |
| --- | --- |
| `case` | 该结果对应的概念请求；常为{kind,value}，材料表中显示legacy:芦笙这样的字符串。 |
| `blocked` | 本案例不能继续的原因；null表示未记录阻塞。对象内stage是发生阶段，reason是类别，detail是可选详细说明。 |
| `result` | 当前这一步的结果对象，可点击展开；具体字段见本表后续各行。 |
| `result.passages` | 本次给文字模型的正文片段列表。 |
| `passages[].source_id / material_id` | 片段ID／它来自哪条材料；source_id供知识引文引用。 |
| `passages[].text` | 实际选中的清洗正文片段。 |
| `passages[].start / end / original_chars` | 片段在清洗正文中的字符范围及清洗正文总字符数；original_chars在这里不是下载原文长度。 |
| `passages[].source_family` | 页面分组用的地址；不同URL变体未必都能归并，不能等同独立来源机构。 |
| `passages[].provenance / document_sha256` | 材料文件定位信息／完整清洗正文的内容哈希。 |
| `passages[].quote_basis / cleaning_version` | 引文核对基准cleaned_text／使用的清洗版本。 |
| `passages[].raw_source_sha256 / source_locator` | 下载文本的哈希和定位；Wiki指保存章节的序列化文本，source_locator说明依据。 |
| `passages[].source_blocks` | 与片段重叠的原文块及原始／清洗范围，留在磁盘供回查，不全部发送模型。 |
| `result.images` | 已拿到可用字节的图片列表；image_id是图片哈希派生标识，record和bytes保留来源与检查。 |
| `result.duplicates` | 重复材料；same_text_as或same_bytes_as指保留的对应材料ID，provenance保留重复项出处。 |
| `result.omissions` | 未选／截断的文档范围及reason；如有start/end，表示未读文字的范围。 |
| `result.image_gaps` | 没有准备好字节的图片及失败状态。 |
| `result.coverage` | 对本次选择范围的说明，不宣称全概念覆盖。 |
| `image_gaps[].bytes.status` | 除前述状态外：remote_not_configured=未配置远程地址；invalid_locator=定位非法；remote_error=远程失败；byte_budget_exceeded=超过下载字节预算；changed_or_missing_bytes=已核验文件变化或消失。 |
| `bytes.http_status / url / acquisition_url` | 若有：远程HTTP状态／尝试地址／成功获取地址。 |


In [ ]:
show([{'case':r['bundle']['request'],'blocked':r.get('blocked'),'result':r.get('material_pack')} for r in organized_rows], limit=100)
if organized_rows: detail(organized_rows[0].get('material_pack', organized_rows[0].get('blocked')))

## 9．从多篇正文提取带引文的知识候选 · ExtractKnowledge

**任务：让模型共同阅读本次所选文字，产出可回查的知识候选。** 比如来源明确支持某种乐器结构时，候选要保留适用范围和引文，不能直接写成试题答案指令。

- **输入到算子**：`organized_rows`。**实际给模型**：目标概念标签，以及片段的`source_id/text/source_family/start/end`。不传图片、未选章节或历史模型结论。
- **实际操作**：联合比较片段中的重复、互补及矛盾，提出最多6条知识候选；程序核对字段、来源ID和引文是否为输入片段的连续子串。
- **输出**：`extracted_rows`，新增`extraction`：`facts`（陈述、条件、例外和引文）、`unresolved_conflicts`（待解差异与缺证）、`coverage_note`；另存完整请求与响应。
- **交给下一步**：候选连同原来的同一批片段交给consolidate复核。

**本批状态：尚未调用模型。** 最多6条是当前工程上限，不是长期知识库每概念应有的条数。

**怎么看**：从候选回查来源是否真支持，再从来源反查重要内容是否漏掉。引文匹配的是清洗片段；匹配成功不证明网站可靠、解释正确或条件完整。

### 本算子的输入字段

| 输入字段／参数 | 含义 | 从哪里来 |
| --- | --- | --- |
| `organized_rows[].identity.target_label` | 供模型理解任务的目标概念名称。 | 第7步 |
| `organized_rows[].material_pack.passages` | 已选中的正文片段列表。 | 第8步 |
| `passages[].source_id` | 本次片段标识，模型引文必须引用它。 | 第8步生成 |
| `passages[].text` | 实际送入模型的清洗文字。 | 第8步按预算截取 |
| `passages[].source_family` | 页面分组地址，帮助保留来源信息，不是可靠性评分。 | 第8步 |
| `passages[].start / end` | 片段在清洗正文中的左闭右开字符范围。 | 第8步 |
| `passages中的其他来源字段` | 原始块、哈希和定位仍保留给程序与审核，但不传给模型。 | 第8步 |

**模型实际输入**为`concept`（target_label）和`passages`；每个片段仅发送上面列出的source_id、text、source_family、start、end。不发送图片。通用模型参数见配置字段说明。

In [ ]:
extracted_rows = []
if (MODE == 'view_saved' or RUN_MODEL_CELLS) and organized_rows:
    extracted_rows = await session.knowledge(ExtractKnowledge, organized_rows)
else:
    print('未执行：模型调试开关关闭，或前一步没有输出。')

### 本表字段：提取的知识候选

| 字段 | 含义 |
| --- | --- |
| `case` | 该结果对应的概念请求；常为{kind,value}，材料表中显示legacy:芦笙这样的字符串。 |
| `blocked` | 本案例不能继续的原因；null表示未记录阻塞。对象内stage是发生阶段，reason是类别，detail是可选详细说明。 |
| `result` | 当前这一步的结果对象，可点击展开；具体字段见本表后续各行。 |
| `result.facts` | 知识候选列表，空列表表示没有候选；与阶段未执行导致result为空不同。 |
| `facts[].fact_id` | 本次结果内的知识标识，如F1；不是全库永久知识ID。 |
| `facts[].statement` | 候选知识陈述。 |
| `facts[].conditions / exceptions` | 适用条件／例外列表。 |
| `facts[].evidence` | 文字证据列表；每项source_id指向输入片段，quote是该片段中的逐字连续引文。 |
| `result.unresolved_conflicts` | 未解决的差异／冲突列表；source_ids是相关片段，issue描述争议，needed_evidence说明缺什么证据。 |
| `result.coverage_note` | 模型对本轮片段范围、未处理内容的说明。 |

原始调用信息另存在extraction_call，其路径、模型、耗时和usage含义同第7步call。本批尚未调用。

In [ ]:
show([{'case':r['bundle']['request'],'blocked':r.get('blocked'),'result':r.get('extraction')} for r in extracted_rows], limit=100)
if extracted_rows: detail(extracted_rows[0].get('extraction', extracted_rows[0].get('blocked')))

## 10．对照原文修订候选，记录未解决冲突 · ConsolidateKnowledge

**任务：再检查本次多材料提取是否误读、漏条件或把未解争议写成确定结论。**

- **输入**：`extracted_rows`中的候选与原来的`material_pack.passages`。实际模型仍是同一本地Qwen，看到的是同一批片段。
- **实际操作**：复查候选，记录保留、修订、删除或合并的理由；要求未解决的实质争议放入冲突记录。程序再次检查结构和逐字引文，并检查存在变更列表。
- **输出**：`consolidated_rows`，新增`knowledge`（修订后的facts、未解冲突、覆盖说明、`changes`）及机器候选审核状态。
- **交给下一步**：evidence使用复核后的facts；export同时保留facts与未解冲突。

**本批状态：尚未调用模型。** 此步不补读全文、不自动找新证据，也不是全库一致性检查。

**怎么看**：最需要查“冲突列表里暂缓的说法是否仍留在facts里”，以及所谓条件差异是否真有原文支持。现有程序检查不能保证冲突已经在语义上解决；同模型再次同意不是独立审核，更不是人工确认。

### 本算子的输入字段

| 输入字段／参数 | 含义 | 从哪里来 |
| --- | --- | --- |
| `extracted_rows[].identity.target_label` | 当前概念名称。 | 第7步保留 |
| `extracted_rows[].material_pack.passages` | 仍是原先选定的同一批正文，不扩展为全文。 | 第8步保留 |
| `extracted_rows[].extraction.facts` | 上一轮候选陈述、条件、例外、source_id及quote；需要对照原文检查。 | 第9步模型输出 |
| `extracted_rows[].extraction.unresolved_conflicts` | 上一轮发现的未解决差异及缺证说明。 | 第9步 |
| `extracted_rows[].extraction.coverage_note` | 上一轮对所处理材料范围的说明。 | 第9步 |

**模型实际输入**为`concept`、`passages`和`candidates`。passages字段与第9步相同；candidates是完整extraction对象，不是独立事实证据。模型未获得新来源或新图片。

In [ ]:
consolidated_rows = []
if (MODE == 'view_saved' or RUN_MODEL_CELLS) and extracted_rows:
    consolidated_rows = await session.knowledge(ConsolidateKnowledge, extracted_rows)
else:
    print('未执行：模型调试开关关闭，或前一步没有输出。')

### 本表字段：对照原文后的候选

| 字段 | 含义 |
| --- | --- |
| `case` | 该结果对应的概念请求；常为{kind,value}，材料表中显示legacy:芦笙这样的字符串。 |
| `blocked` | 本案例不能继续的原因；null表示未记录阻塞。对象内stage是发生阶段，reason是类别，detail是可选详细说明。 |
| `result` | 当前这一步的结果对象，可点击展开；具体字段见本表后续各行。 |
| `result.facts` | 知识候选列表，空列表表示没有候选；与阶段未执行导致result为空不同。 |
| `facts[].fact_id` | 本次结果内的知识标识，如F1；不是全库永久知识ID。 |
| `facts[].statement` | 候选知识陈述。 |
| `facts[].conditions / exceptions` | 适用条件／例外列表。 |
| `facts[].evidence` | 文字证据列表；每项source_id指向输入片段，quote是该片段中的逐字连续引文。 |
| `result.unresolved_conflicts` | 未解决的差异／冲突列表；source_ids是相关片段，issue描述争议，needed_evidence说明缺什么证据。 |
| `result.coverage_note` | 模型对本轮片段范围、未处理内容的说明。 |
| `result.changes` | 模型记录的修订列表。 |
| `changes[].fact_id / action / reason` | 对应候选标识／kept保留、revised修订、removed删除、merged合并／处理理由。 |
| `knowledge_review_status（完整行中）` | 明确标为机器候选，语义正确性和冲突处理等待审核。 |
| `consolidation_call（完整行中）` | 复核调用的路径、模型、token与耗时；含义同第7步call。 |


In [ ]:
show([{'case':r['bundle']['request'],'blocked':r.get('blocked'),'result':r.get('knowledge')} for r in consolidated_rows], limit=100)
if consolidated_rows: detail(consolidated_rows[0].get('knowledge', consolidated_rows[0].get('blocked')))

## 11．逐图逐知识判定支持范围 · CheckImageSupport

**任务：检查具体图片能证明哪些知识，以及不能证明哪些内容。** 同概念图片未必能支持结构、连接或机制知识。

- **输入到算子**：`consolidated_rows`中的facts与`material_pack.images`。**实际给模型**：facts及图片像素，不重新输入网页全文或已有caption。
- **实际操作**：复验原图哈希，修正EXIF方向，生成最长边不超过1,024像素的JPEG模型输入；记录原图与输入图哈希。模型描述可见内容，并逐图逐知识判断支持程度；程序检查图片×知识组合是否完整覆盖。
- **输出**：`evidence_rows`，新增`image_evidence`。实际调用后含图片描述、支持状态`full/partial/none/unobservable`、区域`region`、支持内容`supports`及局限`limitations`。无图或无facts则记录`not_run`与原因。
- **交给下一步**：图片支持关系与知识一起保存，不把模型描述覆盖到来源图注。

**本批状态：尚未执行；真实图像模型分支尚未验收。** COS真实路由也不能因已有适配代码就称为可用。

**怎么看**：确认部位是否真的可见、图片缩放是否损失细节、支持范围是否夸大。元数据和caption仅是线索，不能替代此处实际像素检查；机器支持结论也需审核。

### 本算子的输入字段

| 输入字段／参数 | 含义 | 从哪里来 |
| --- | --- | --- |
| `consolidated_rows[].knowledge.facts` | 复核后的知识候选，含陈述、条件、例外与文字引文。 | 第10步输出 |
| `consolidated_rows[].material_pack.images` | 已准备好字节的图片，不包括image_gaps中的失败记录。 | 第8步保留 |
| `images[].image_id` | 图片标识，供模型返回逐图支持关系。 | 第8步生成 |
| `images[].bytes.path` | 读取实际图片文件的绝对路径。 | 图片字节检查／获取步骤 |
| `images[].record.sha256` | 再核对文件是否仍是选定图片的预期哈希。 | 采集记录 |
| `像素预处理参数` | 按EXIF旋转、转RGB、最长边1,024像素、JPEG质量90；原图不覆盖。 | CheckImageSupport实现 |

**模型实际输入**为facts文本、各图片image_id及预处理后的像素。source caption、网页全文不另行发送。facts中带有原先文字引文，故这一步不是“仅凭图片识别知识”，而是检查图片能否支持给定知识。

In [ ]:
evidence_rows = []
if (MODE == 'view_saved' or RUN_MODEL_CELLS) and consolidated_rows:
    evidence_rows = await session.knowledge(CheckImageSupport, consolidated_rows)
else:
    print('未执行：模型调试开关关闭，或前一步没有输出。')

### 本表字段：图片对知识的支持

| 字段 | 含义 |
| --- | --- |
| `case` | 该结果对应的概念请求；常为{kind,value}，材料表中显示legacy:芦笙这样的字符串。 |
| `blocked` | 本案例不能继续的原因；null表示未记录阻塞。对象内stage是发生阶段，reason是类别，detail是可选详细说明。 |
| `result` | 当前这一步的结果对象，可点击展开；具体字段见本表后续各行。 |
| `result.status` | not_run=无图或无知识而未调用；machine_reviewed=模型已给出支持判断，不是人工审核。 |
| `result.reason` | 未调用原因：no_available_images或no_knowledge_candidates。 |
| `result.result.images` | 真正调用后返回的图片描述列表；image_id标识图片，caption是本次模型可见内容描述。注意存在外层表格result和内层模型result两层。 |
| `result.result.support` | 逐图逐知识的支持判断列表。 |
| `support[].image_id / fact_id` | 这条判断针对哪张图片、哪条知识。 |
| `support[].status` | full完整支持／partial部分支持／none不支持／unobservable无法观察；均为模型判断。 |
| `support[].region / supports / limitations` | 图中区域／能支持的内容／不能证明的内容及局限。 |
| `result.image_roles` | 每张实际输入图的记录：image_id、原图path、original_sha256、input_sha256和role。两种哈希分别对应原图与缩放重编码后的模型输入。 |
| `result.note` | 本次模型描述与人工审核／来源图注的边界说明。 |
| `result.call` | 调用信息，字段含义同第7步。 |
| `result.images / result.support（not_run时）` | 未调用分支直接保存的空列表；不是模型做出了零支持判断。 |


In [ ]:
show([{'case':r['bundle']['request'],'blocked':r.get('blocked'),'result':r.get('image_evidence')} for r in evidence_rows], limit=100)
if evidence_rows: detail(evidence_rows[0].get('image_evidence', evidence_rows[0].get('blocked')))

## 12．保存知识候选、图片支持关系与阻塞记录 · ExportCandidates

**任务：把本次能产出的候选和不能继续的原因都保存下来，方便复核与后续版本比较。**

- **输入**：`evidence_rows`及其保留的身份、知识、冲突与阻塞状态。即使前面阻塞，export仍保存记录。
- **实际操作**：整理并不可覆盖地写入`candidates/<case_id>.json`，不调用模型、不修改datasets或评分。
- **输出**：`exported_rows`中的`export`，包含概念ID、请求、身份结果、facts、未解冲突、图片支持和状态。状态为`blocked`或`machine_candidates_ready_for_review`。
- **后续使用**：通过来源、条件、争议及支持关系的审核后，才可能成为可用知识资产；是否进入V4另作任务适配核验。

**本批状态：尚未执行。** 当前导出的文件只是候选交付，完整来源映射仍需联合查看阶段记录；它不是已经完成审核的干净知识库查询服务。

**怎么看**：查看是否把机器候选误认为已核验知识；即使状态是ready，也要核对未解冲突、缺图和检查范围。保存成功不代表质量合格。

### 本算子的输入字段

| 输入字段／参数 | 含义 | 从哪里来 |
| --- | --- | --- |
| `evidence_rows[].case_id / bundle.concept_id / bundle.request` | 要保存的案例、内部概念ID和原始请求。 | 前面各阶段保留 |
| `evidence_rows[].blocked` | 若前面无法继续，保存具体原因。 | 任一前置阶段 |
| `evidence_rows[].identity` | 本次身份判断及调用依据。 | 第7步 |
| `evidence_rows[].knowledge.facts / unresolved_conflicts` | 拟保存的知识候选与未解决冲突；缺少knowledge时按空列表保存。 | 第10步 |
| `evidence_rows[].image_evidence` | 图片支持判断或未调用原因；尚未产生时为null。 | 第11步 |
| `RUN` | 写入candidates/<case_id>.json的位置。 | 配置cell |

此步不调用模型。它不接收新的审核决定，也不会把ready状态升级成人工已核验。

In [ ]:
exported_rows = []
if (MODE == 'view_saved' or RUN_MODEL_CELLS) and evidence_rows:
    exported_rows = await session.knowledge(ExportCandidates, evidence_rows)
else:
    print('未执行：模型调试开关关闭，或前一步没有输出。')

### 本表字段：保存的候选与阻塞

| 字段 | 含义 |
| --- | --- |
| `case` | 该结果对应的概念请求；常为{kind,value}，材料表中显示legacy:芦笙这样的字符串。 |
| `blocked` | 本案例不能继续的原因；null表示未记录阻塞。对象内stage是发生阶段，reason是类别，detail是可选详细说明。 |
| `result` | 当前这一步的结果对象，可点击展开；具体字段见本表后续各行。 |
| `result.case_id` | 绑定本批材料输入的案例标识。 |
| `result.concept_id` | 独立试运行注册表中的内部概念ID；不表示已完成全库身份合并。 |
| `result.request` | 原始概念名／QID请求。 |
| `result.status` | blocked=阻塞；machine_candidates_ready_for_review=保存了待审核候选。 |
| `result.blocked` | 阻塞步骤、类别与详情，含义同外层blocked。 |
| `result.identity` | 第7步身份判断的完整结果。 |
| `result.facts / unresolved_conflicts` | 第10步保存的知识候选／未解冲突，字段同第9、10步。 |
| `result.image_evidence` | 第11步图片支持结果或未执行原因。 |
| `result.scope` | 明确导出不是人工金标准、试题或完整概念知识库。 |


In [ ]:
show([{'case':r['bundle']['request'],'blocked':r.get('blocked'),'result':r.get('export')} for r in exported_rows], limit=100)
if exported_rows: detail(exported_rows[0].get('export', exported_rows[0].get('blocked')))

## 13．读取已保存的步骤结果与模型请求

**任务：只读复盘运行证据，不执行业务算子。**

- **输入**：`RUN`下已保存文件的路径。
- **实际操作**：列出步骤文件；用`read`和`detail`展开完整记录，必要时查看模型实际发送的请求与原始响应。
- **输出**：notebook中的表格／全文展示，不产生新材料或审核结论。

| 运行目录内的位置 | 保存什么 |
| --- | --- |
| `debug_manifest.json` | 调试运行的代码、notebook代码cell、配置与依赖版本 |
| `debug_steps/<step>.json` | 各cell对应的输入哈希及完整输出，可只读恢复 |
| `source_tasks/` | 来源扫描缓存、命中记录与覆盖范围 |
| `bundles/` | 冻结的原始材料包 |
| `stages/<stage>/` | clean及知识算子的逐案例状态与输出 |
| `calls/` | 如实际调用模型，保存完整请求、响应与失败信息 |
| `candidates/` | 如已执行export，保存候选或阻塞记录 |

**本批实际**：已保存6个调试步骤（检查、两次扫描、组装、保存、清洗），模型请求数为0。缺少后半段输出表示未执行，不是“执行后没有发现知识”。

**怎么看**：`MODE='view_saved'`只读取历史结果，忽略当前执行配置而展示当时版本；要确认当时参数请看manifest。`MODE='run'`才执行或按冻结规则复用算子。显示采样种子、limit及字段展开不构成业务重跑。

### 本算子的输入字段

| 输入字段／参数 | 含义 | 从哪里来 |
| --- | --- | --- |
| `RUN / debug_steps/*.json` | 要列出的历史步骤文件。 | 配置cell指定运行目录 |
| `limit / sample / seed / chars` | 只控制显示多少条、是否抽样、抽样种子和折叠预览长度。 | show调用参数 |
| `read(path)中的path` | 需要展开的某个已有步骤、请求或响应文件路径。 | 文件列表或调用记录 |
| `detail(row)中的row` | 想完整查看的Python对象；可为单条材料、阶段结果或请求。 | 上一步输出或read返回值 |

只读取已有数据并显示，不生成业务输入、不验证当前配置与历史结果一致。核对真实执行输入时，应查看该运行的manifest和calls。

### 本表字段：步骤输出文件

| 字段 | 含义 |
| --- | --- |
| `step` | 保存的步骤名，例如clean、scan_materials。 |
| `path` | 步骤JSON文件的绝对路径。 |
| `bytes` | 此处是文件大小，单位字节；与材料表中的图片检查对象bytes不同。 |
| `已保存模型请求数` | calls目录下request.json文件数；包括已发起但可能缺响应的请求，不等于HTTP成功数。 |

展开步骤JSON时：input_hash绑定输入与配置，rows是该步完整输出。模型请求文件含stage、endpoint及payload；响应文件含status_code、elapsed_s和body。

In [ ]:
show([{'step':p.stem,'path':str(p),'bytes':p.stat().st_size} for p in sorted((RUN/'debug_steps').glob('*.json'))])
print('已保存模型请求数:', len(list((RUN/'calls').glob('*.request.json'))))
# from curation.v4.contracts import read
# detail(read(RUN/'debug_steps/identity.json')['rows'][0])
# detail(read(next((RUN/'calls').glob('*.request.json'))))